In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 1. --- Global Settings ---

# Set global font and style for a professional look
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 24,
    'axes.titlesize': 24,
    'axes.labelsize': 24,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'legend.fontsize': 48,
    'figure.titlesize': 24
})

# Define the mapping from CSV columns directly to publication-quality names
column_mapping = {
    'renewable_cost_per_capita': "Renewable Generation Cost",
    'storage_cost_per_capita': "Energy Storage System Cost",
    'lng_cost_per_capita': "Conventional Generation Cost",
    'other_equipment_cost_per_capita': "Sector-Coupling System Cost",
    'discard_cost_per_capita': "Penalty Cost of Curtailment",
    'load_shedding_cost_per_capita': "Cost of Unserved Energy"
}

# Define the color for each cost type, using the publication-quality names as keys
cost_color_map = {
    "Renewable Generation Cost": "#4C72B0",
    "Energy Storage System Cost": "#DDAA33",
    "Conventional Generation Cost": "#808080",
    "Sector-Coupling System Cost": "#8172B3",
    "Penalty Cost of Curtailment": "#d47d49",
    "Cost of Unserved Energy": "#C44E52"
}

# 2. --- Data Loading and Processing ---

# Load the island cost data
real_cost_df = pd.read_csv('../result/island_cost_summary_0.csv')

# Filter for regions with sufficient data
region_counts = real_cost_df['IPCC_Region_Code'].value_counts()
valid_regions = region_counts[region_counts >= 10].index.tolist()
filtered_df = real_cost_df[real_cost_df['IPCC_Region_Code'].isin(valid_regions)].copy()

# Rename columns for internal use using the direct mapping
for old_col, new_col in column_mapping.items():
    if old_col in filtered_df.columns:
        filtered_df[new_col] = filtered_df[old_col]

# Sort regions by their mean total per-capita cost
cost_names = list(column_mapping.values())
filtered_df['total_cost_per_capita'] = filtered_df[cost_names].sum(axis=1)
total_cost_per_region = filtered_df.groupby('IPCC_Region_Code')['total_cost_per_capita'].mean()
areas = total_cost_per_region.sort_values(ascending=False).index.tolist()

# Rename region column for plotting
filtered_df['Area'] = filtered_df['IPCC_Region_Code']

print(f"Using data from {len(filtered_df)} islands.")
print(f"Regions included: {areas}")
print(f"Cost types (formal names): {cost_names}")

# 3. --- 统一x轴的云雨图组合 ---

# 计算全局x轴范围
all_values = []
for cost_name in cost_names:
    all_values.extend(filtered_df[cost_name].dropna().values)
global_min = min(all_values)
global_max = max(all_values)
x_margin = (global_max - global_min) * 0.05
xlim = (global_min - x_margin, global_max + x_margin)

# 创建垂直排列的子图，共享x轴
fig, axes = plt.subplots(nrows=len(cost_names), ncols=1, figsize=(16, 24), 
                         sharex=True, dpi=300)

# 确保axes是数组格式
if len(cost_names) == 1:
    axes = [axes]

# 为每个成本类型生成子图
for idx, cost_name in enumerate(cost_names):
    ax = axes[idx]
    current_color = cost_color_map[cost_name]
    
    # 准备当前成本类型的数据
    area_groups = filtered_df.groupby('Area')[cost_name].apply(list)
    area_groups = area_groups.reindex(areas)
    
    # 定义x轴上的位置
    positions = np.arange(len(areas)) + 1
    
    # 为每个区域创建云雨图
    for i, area in enumerate(areas):
        if area not in area_groups.index or len(area_groups[area]) == 0:
            continue
            
        # A. 小提琴图（云）
        violin_parts = ax.violinplot(
            area_groups[area], positions=[positions[i]],
            showmeans=False, showmedians=False, showextrema=False
        )
        
        # 设置小提琴图样式
        for pc in violin_parts['bodies']:
            pc.set_facecolor(current_color)
            pc.set_edgecolor('black')
            pc.set_alpha(0.8)

        # B. 箱线图（雨）
        ax.boxplot(
            area_groups[area], positions=[positions[i]], widths=0.18,
            patch_artist=True,
            boxprops=dict(facecolor='white', color='black'),
            medianprops=dict(color='#be1420', linewidth=2),
            whiskerprops=dict(color='black', linewidth=1),
            capprops=dict(color='black', linewidth=1),
            flierprops=dict(marker='o', color='black', markersize=3, markerfacecolor='gray'),
            showfliers=True
        )

        # C. 散点图（雨滴）
        # jittered_x = np.random.normal(positions[i] - 0.15, 0.03, size=len(area_groups[area]))
        # ax.scatter(
        #     jittered_x, area_groups[area], color=current_color,
        #     alpha=0.7, s=15, edgecolors='black', linewidths=0.3
        # )

    # 设置子图样式
    ax.set_ylabel('Cost (USD capita$^{-1}$)', fontsize=24)
    ax.set_xlim(0.5, len(areas) + 0.5)
    
    # 设置x轴刻度和标签（只在最后一个子图显示）
    if idx == len(cost_names) - 1:
        ax.set_xticks(positions[:len(areas)])
        ax.set_xticklabels(areas, rotation=45, ha='right')
    else:
        ax.set_xticks([])
    
    # 美化子图
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)

fig.align_ylabels(axes)

# 调整子图之间的间距
plt.subplots_adjust(hspace=0.9)
plt.tight_layout()
plt.show()

# 4. --- 生成统一的图例 ---

# 创建图例
fig_legend, ax_legend = plt.subplots(figsize=(16, 8), dpi=300)

legend_handles = [mpatches.Patch(color=color, label=label) 
                  for label, color in cost_color_map.items()]

fig_legend.legend(
    handles=legend_handles, 
    loc='center', 
    frameon=False, 
    ncol=3,
    fontsize=48
)

ax_legend.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 6. --- 各成本类型的区域平均值排序可视化 ---

# 计算每个成本类型在各区域的平均值
cost_means = {}
for cost_name in cost_names:
    regional_means = filtered_df.groupby('Area')[cost_name].mean().sort_values(ascending=False)
    cost_means[cost_name] = regional_means

# 为每个成本类型创建单独的图，但保持x轴长度一致
for idx, cost_name in enumerate(cost_names):
    # 创建单独的图
    fig, ax = plt.subplots(figsize=(12, 3), dpi=300)
    current_color = cost_color_map[cost_name]
    
    # 获取当前成本类型的排序数据
    regional_means = cost_means[cost_name]
    
    # 创建垂直柱状图
    bars = ax.bar(range(len(regional_means)), regional_means.values, 
                  color=current_color, alpha=0.7, edgecolor='black')
    
    # 设置x轴标签 - 确保所有图都显示相同数量的区域
    ax.set_xticks(range(len(areas)))  # 使用areas确保x轴长度一致
    ax.set_xticklabels(regional_means.index, fontsize=18, rotation=45, ha='right')
    
    # 设置y轴标签
    ax.set_ylabel('Cost', fontsize=18)
    
    # 在柱状图上显示数值
    # for i, (area, value) in enumerate(regional_means.items()):
    #     ax.text(i, value + max(regional_means) * 0.02, f'{value:.2f}', 
    #             ha='center', va='bottom', fontsize=12)
    
    # 美化图表
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    
    # 设置y轴范围
    ax.set_ylim(0, max(regional_means) * 1.15)
    
    # 确保x轴长度一致：设置相同的x轴范围
    ax.set_xlim(-0.5, len(areas) - 0.5)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 7. --- 基于IPCC区域的分层设色世界地图 ---

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap

# 读取IPCC区域地理数据
ipcc_regions = gpd.read_file("IPCC-WGI-reference-regions-v4.geojson")

# 为每个成本类型创建世界地图
for cost_name in cost_names:
    fig = plt.figure(figsize=(14, 8), dpi=300, facecolor='none')
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

    # 设置背景和地图特征
    # ax.set_facecolor('#CCCCCC')
    ax.set_facecolor('none')
    ax.add_feature(cfeature.LAND, color="#CECECE", alpha=0.4)
    ax.add_feature(cfeature.OCEAN, color="#FFFFFF", alpha=0.5)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, alpha=0.3)
    
    # 获取当前成本类型的区域平均值
    regional_means = filtered_df.groupby('Area')[cost_name].mean()
    
    # 创建颜色映射
    if len(regional_means) > 0:
        vmin, vmax = regional_means.min(), regional_means.max()
        norm = Normalize(vmin=vmin, vmax=vmax)
        
        # 使用与前面相同的成本类型颜色
        base_color = cost_color_map[cost_name]
        # 创建从白色到该颜色的渐变
        colors = ['white', base_color]
        colormap = LinearSegmentedColormap.from_list('custom', colors, N=256)
        
        if ipcc_regions is not None and not ipcc_regions.empty:
            # 遍历每个IPCC区域
            for _, region_row in ipcc_regions.iterrows():
                region_code = region_row['Acronym']  # 或者使用适当的列名
                
                if region_code in regional_means.index:
                    # 获取该区域的成本值和对应颜色
                    cost_value = regional_means[region_code]
                    color = colormap(norm(cost_value))
                    
                    # 绘制填充的区域
                    ax.add_geometries([region_row['geometry']], crs=ccrs.PlateCarree(),
                                      facecolor=color,
                                      edgecolor='white',
                                      linewidth=0.8,
                                      alpha=0.8,
                                      zorder=2)
                else:
                    # 如果该区域没有数据，用灰色填充
                    ax.add_geometries([region_row['geometry']], crs=ccrs.PlateCarree(),
                                      facecolor='white',
                                      edgecolor='white',
                                      linewidth=0.8,
                                      alpha=0,
                                      zorder=2)
    
    # 设置视图范围，排除南极洲
    ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
    # --- Longitude/latitude graticule (Nature Comms checklist) ---
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                      linewidth=0.4, color='gray', alpha=0.5, linestyle='--',
                      xlocs=range(-180, 181, 60), ylocs=range(-60, 91, 30))
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 24, 'color': '#404040'}
    gl.ylabel_style = {'size': 24, 'color': '#404040'}
    
    # 去除所有标签和图例
    ax.set_xticks([])
    ax.set_yticks([])
    
    # 去除边框
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # 去除所有边距，只显示图像内容
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.axis('off')
    
    plt.show()

In [ ]:
# --- Standalone colour-scale key for the Fig. 1b region cost maps ---
# Every panel shades IPCC regions white -> colour by the regional mean per-capita
# cost (darker = higher), each on its own range. A single schematic bar conveys
# that shared low->high meaning.
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize, LinearSegmentedColormap

fig_cb = plt.figure(figsize=(5, 1.1), dpi=300)
cax = fig_cb.add_axes([0.37, 0.55, 0.26, 0.16])   # short thin horizontal band

cmap = LinearSegmentedColormap.from_list('shade', ['white', '#404040'], N=256)
sm = cm.ScalarMappable(norm=Normalize(vmin=0, vmax=1), cmap=cmap)
sm.set_array([])

cb = fig_cb.colorbar(sm, cax=cax, orientation='horizontal')
cb.set_ticks([0, 1])
cb.ax.set_xticklabels(['Low', 'High'])
cb.ax.tick_params(labelsize=11, length=0)
cb.outline.set_visible(False)
cb.set_label('Per-capita cost', fontsize=12, labelpad=1)

fig_cb.savefig('fig1b_region_cost_colorbar.png', dpi=300, bbox_inches='tight')
plt.show()
